# RANK3：四新源阶段2确认

直接 Run all。默认下载公开 B 分支 dev/local-joint-state-payload-v1 可用源码到隔离目录；无需源码 ZIP。第一代码单元仅挂载 Drive。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 输入与只读恢复

旧输入 fixed-four-new-v1 仅提供已冻结背景和 τ=0.49221603782894896。四新源的 prompt、seed、a 和 eval 角色固定。新输出 B-Rank3-Four-Source-Confirmation-V1/fixed-four-new-v1；已有 attempt 时在下载与 setup 前只读恢复，不续跑或补缺。模型与 HF 缓存只放本次会话 /content/video-wm-cache；RGB、latent、DCT 临时张量放会话 /content，完成或记录失败后移除。Drive 保留 JSON 证据与最终 MP4；会话丢失不补跑。


In [ ]:

from pathlib import Path
import json, os, shutil, stat, subprocess, sys, tempfile, urllib.request, zipfile
os.environ.update(HF_HOME='/content/video-wm-cache/huggingface', HF_HUB_CACHE='/content/video-wm-cache/huggingface/hub', TRANSFORMERS_CACHE='/content/video-wm-cache/huggingface/hub', TORCH_HOME='/content/video-wm-cache/torch')
INPUT = Path('/content/drive/MyDrive/Video-WM/B-Terminal-Three-Group-V1/fixed-four-new-v1')
OUTPUT = Path('/content/drive/MyDrive/Video-WM/B-Rank3-Four-Source-Confirmation-V1/fixed-four-new-v1')
SOURCE_REF = 'dev/local-joint-state-payload-v1'
SOURCE_DIRECTORY = None  # Optional usable local tree; copied into an isolated directory.
KEYS={'key00':'watermark','key01':'watermark-wrong'}

ARMS=('OFF','P0','PM')

WINDOWS={'L':0,'M':53,'R':106}

FIXTURES={'CROP':(0,None),'DELETE_LEFT_GAP':(8,49),'DELETE_RIGHT_GAP':(8,102)}

SOURCES=[{'id': 'b-rank3-n01', 'role': 'eval', 'seed': 2026102501, 'a': 2, 'prompt': 'A fixed wide shot of a red tram moving slowly along a tree-lined avenue. Pedestrians wait beside the track and leaves move gently in the breeze.', 'negative_prompt': ''}, {'id': 'b-rank3-n02', 'role': 'eval', 'seed': 2026102502, 'a': 11, 'prompt': 'A locked-off medium-wide shot of a person folding a blue cotton cloth on a workbench. Both hands and the entire cloth remain visible throughout the shot.', 'negative_prompt': ''}, {'id': 'b-rank3-n03', 'role': 'eval', 'seed': 2026102503, 'a': 17, 'prompt': 'A fixed wide shot of a small waterfall flowing over mossy rocks into a shallow forest stream. Water ripples and nearby fern leaves move naturally.', 'negative_prompt': ''}, {'id': 'b-rank3-n04', 'role': 'eval', 'seed': 2026102504, 'a': 6, 'prompt': 'A fixed wide shot of a white wind turbine rotating slowly above a grassy meadow under moving clouds. The whole turbine and foreground grass remain in frame.', 'negative_prompt': ''}]

def task_id(h, arm, fixture, key):
    return '/'.join((h, arm, fixture, key))

def recover_ledger(output):
    """Read-only receipt overlay; also embedded verbatim in the notebook.

    STARTED remains spent and incomplete. Invalid or uncommitted receipts
    remain unresolved, never an inferred successful or unentered call.
    """
    from collections import Counter
    import json
    from pathlib import Path
    output = Path(output)
    ledger={slot:dict(slot=slot,kind=kind,status='NOT_ENTERED',attempts=0) for slot,kind in plan_units().items()}
    errors, receipt_slots = {}, []
    try:
        saved = json.loads((output/'ledger.json').read_text(encoding='utf-8'))
        if not isinstance(saved,dict):raise ValueError('ledger must map slots to records')
        for slot,row in saved.items():
            if slot not in ledger:
                errors['ledger/'+slot]='unknown saved slot';continue
            if not isinstance(row,dict) or row.get('kind')!=ledger[slot]['kind']:
                ledger[slot].update(status='UNRESOLVED_LEDGER',attempts=1 if isinstance(row,dict) and row.get('attempts')==1 else None)
                errors['ledger/'+slot]='invalid saved slot';continue
            status,attempts=row.get('status'),row.get('attempts')
            if status not in ('NOT_ENTERED','STARTED','RETURNED','FAILED','INTERRUPTED') or type(attempts) is not int or attempts!=(0 if status=='NOT_ENTERED' else 1):
                ledger[slot].update(status='UNRESOLVED_LEDGER',attempts=1 if attempts==1 else None);errors['ledger/'+slot]='invalid status/attempt';continue
            ledger[slot]=row
    except (OSError, ValueError) as error:
        errors['ledger.json'] = str(error)
    directory = output/'receipts'
    paths = sorted(directory.rglob('*.json')) + sorted(directory.rglob('*.json.tmp'))
    for path in paths:
        relative = path.relative_to(directory).as_posix()
        pending = relative.endswith('.tmp')
        slot = relative[:-9] if pending else relative[:-5]
        if slot not in ledger:
            errors['receipts/'+relative]='receipt outside fixed plan';continue
        previous = ledger[slot]
        receipt = None
        try:
            receipt = json.loads(path.read_text(encoding='utf-8'))
            if pending:
                raise ValueError('uncommitted receipt; completion is unresolved')
            if not isinstance(receipt, dict) or receipt.get('slot') != slot:
                raise ValueError('receipt slot does not match its address')
            status, attempts = receipt.get('status'), receipt.get('attempts')
            if status not in ('NOT_ENTERED', 'STARTED', 'RETURNED', 'FAILED', 'INTERRUPTED'):
                raise ValueError('invalid receipt status')
            if type(attempts) is not int or attempts != (0 if status == 'NOT_ENTERED' else 1):
                raise ValueError('invalid receipt attempt count')
            if not isinstance(receipt.get('kind'), str) or not receipt['kind']:
                raise ValueError('receipt kind required')
            if previous.get('kind') is not None and previous['kind'] != receipt['kind']:
                raise ValueError('receipt kind disagrees with saved slot')
            if previous.get('attempts') == 1 and attempts == 0:
                raise ValueError('receipt cannot erase a spent call')
            ledger[slot] = receipt
            receipt_slots.append(slot)
        except (OSError, ValueError) as error:
            name = 'receipts/'+relative
            errors[name] = str(error)
            ledger[slot] = dict(previous, slot=slot, status='UNRESOLVED_RECEIPT',
                                attempts=1 if previous.get('attempts') == 1 or isinstance(receipt,dict) and receipt.get('attempts')==1 else None,
                                saved_status=previous.get('status'), recovery_error=name)
    counts = dict(Counter(row.get('status', 'UNRESOLVED_LEDGER') for row in ledger.values()))
    kinds = sorted({row.get('kind', 'UNKNOWN') for row in ledger.values()})
    unit_counts = {kind: dict(Counter(row.get('status', 'UNRESOLVED_LEDGER') for row in ledger.values()
                                     if row.get('kind', 'UNKNOWN') == kind)) for kind in kinds}
    return dict(ledger=ledger, counts=counts, unit_counts=unit_counts,
                receipt_slots=receipt_slots, read_errors=errors, physical_reentry=False,denominators=denominators())

def denominators():
    return dict(sources=4,eval_sources=4,trajectories=8,native_steps=400,DiT_forward=800,
        local_gradient=100,state_construct=4,D181=12,codec_encode=48,codec_probe=48,codec_decode=48,
        E41=432,Q=864,native_key00_Q=8,H_families=216,H_score_slots=29592,E41_DCT=432,
        E41_key_bits=864,native_DCT=8,native_key_bits=16,tasks_per_reader=72,slots_per_reader=2304,
        positive_tasks=12,positive_slots=384,null_tasks=60,null_slots=1920,wrong_key_PM_tasks=12,
        wrong_key_PM_slots=384,local_CDC_slots=6912,M3_path_classes=69192,M3_seam_aliases=604872,
        quality_records=76,quality_frames=12316,quality_adjacent=12240,
        media_P0_PM_bit_pairs=5760,native_P0_PM_bit_pairs=2816,bundles=0)

def tasks():
    return [task_id(h['id'],a,f,k) for h in SOURCES for a in ARMS for f in FIXTURES for k in KEYS]

def quality_specs():
    records={}
    for host in SOURCES:
        h=host['id']
        for pair in ('OFF_to_P0','P0_to_PM'):
            for scope,layers in [('full',('preclip','rgb8','readback'))]+[(f,('rgb8','readback')) for f in FIXTURES]:
                for layer in layers:records[f'{h}/quality/{scope}/{pair}/{layer}']=(h,scope,pair,layer)
        records[f'{h}/quality/full/OFF_to_PM/readback']=(h,'full','OFF_to_PM','readback')
    return records

def plan_units():
    units={'inputs':'input_semantics','background':'background_load','threshold':'old_frozen_tau',
           'plans_complete':'plan_readback','blind_complete':'blind_readback','post':'post_evaluation'}
    for host in SOURCES:
        h=host['id']
        for k in KEYS:units[h+'/book/'+k]='codebook'
        units[h+'/generation_load']='generation_load';units[h+'/vae_load']='vae_load'
        for a in ('OFF','P0'):units[h+'/generate/'+a]='native50_trajectory'
        for a in ('P0','PM'):
            units[f'{h}/native/{a}/Q']='native_key00_Q'
            units[f'{h}/native/{a}/DCT']='native_DCT'
            for k in KEYS:units[f'{h}/native/{a}/{k}/bits']='native_local_bits'
        units[h+'/construct']='state_construct';units[h+'/native_audit']='native_state_audit'
        for a in ARMS:
            for op,kind in [('D','D181'),('preclip','preclip'),('quantize','quantize'),('encode','codec_encode'),('probe','codec_probe'),('decode','codec_decode')]:
                units[f'{h}/{a}/full/{op}']=kind
            for f in FIXTURES:
                stem=f'{h}/{a}/{f}'
                for op,kind in [('edit','fixture151'),('encode','codec_encode'),('probe','codec_probe'),('decode','codec_decode')]:units[stem+'/'+op]=kind
                for w in WINDOWS:
                    for phase in range(4):
                        view=f'{stem}/{w}/p{phase}';units[view+'/E']='E41';units[view+'/DCT']='E41_DCT'
                        for k in KEYS:
                            units[view+'/'+k+'/Q']='E41_Q';units[view+'/'+k+'/bits']='local_and_Rc_bits'
                    for k in KEYS:units[f'{stem}/{w}/{k}/score']='H137_score'
    for task in tasks():
        units['blind/'+task+'/plan']='blind_plan';units['blind/'+task+'/read']='blind_read'
    for slot in quality_specs():units[slot]='quality_pair'
    return units

def execution_artifacts(output):
    output=Path(output)
    names=('attempt.json','plan.json','ledger.json','inputs.json','protocol.json','background.json','thresholds.json',
           'physical_counts.json','result.json','evaluation.json','planned_outputs.json','plans_complete.json','blind_complete.json','post_truth_access.json','candidate_stop.json')
    found=[n+s for n in names for s in ('','.tmp') if (output/(n+s)).exists() or (output/(n+s)).is_symlink()]
    for name in ('receipts','private','media','observations','families','blind','quality'):
        folder=output/name
        if folder.is_dir():found.extend(str(p.relative_to(output)) for p in folder.rglob('*') if p.is_file() or p.is_symlink())
    return sorted(found)
EXISTING_ARTIFACTS = execution_artifacts(OUTPUT)
SOURCE_ROOT = None
if EXISTING_ARTIFACTS:
    print('已有一次执行记录：仅恢复读取，不下载源码、不安装、不加载模型。')
else:
    workspace=Path(tempfile.mkdtemp(prefix='b-rank3-source-')).resolve()
    if SOURCE_DIRECTORY is None:
        archive=workspace/'source.zip'
        url='https://codeload.github.com/RICHAAARC/SC-SSTW/zip/refs/heads/'+SOURCE_REF
        urllib.request.urlretrieve(url,archive)
        destination=workspace/'source';destination.mkdir()
        with zipfile.ZipFile(archive) as source:
            for member in source.infolist():
                normalized=member.filename.replace('\\','/')
                target=(destination/normalized).resolve()
                if normalized.startswith('/') or any(':' in p for p in normalized.split('/')) or not target.is_relative_to(destination.resolve()) or stat.S_ISLNK(member.external_attr>>16):
                    raise ValueError('unsafe source archive entry: '+member.filename)
            source.extractall(destination)
        entries=list(destination.glob('*/experiments/wan_state_clock/b_rank3_four_source_confirmation_v1_run.py'))
        if len(entries)!=1:raise FileNotFoundError('RANK3 source entry unavailable: '+url)
        SOURCE_ROOT=entries[0].parents[2]
    else:
        original=Path(SOURCE_DIRECTORY).expanduser().resolve()
        if not (original/'experiments/wan_state_clock/b_rank3_four_source_confirmation_v1_run.py').is_file():
            raise FileNotFoundError('RANK3 entry missing in SOURCE_DIRECTORY')
        SOURCE_ROOT=workspace/'source'
        shutil.copytree(original,SOURCE_ROOT,ignore=shutil.ignore_patterns('.git','__pycache__'))
    OUTPUT.mkdir(parents=True,exist_ok=True)
    print('隔离源码目录:',SOURCE_ROOT)
    print('原始输入:',INPUT,'新输出:',OUTPUT)


## 一次固定预算

γ=1；8 trajectories / 400 steps / 800 DiT / 100 local gradients / 4 state；D181=12、E41=432、codec encode/probe/decode各48。无新校准、搜索或 bundles。五读法共享观测：72 tasks，每读法 384 positive + 1920 null 槽。全部 plans 回读后才消费缓存位，全部 blind decisions 回读后才做真值评价。首技术失败停止并保留未进入项。


In [ ]:
if EXISTING_ARTIFACTS:
    print('只读已有执行记录。')
else:
    LOG = OUTPUT/'setup.log'
    def command(args, label, check=True):
        print(label, flush=True)
        with LOG.open('a') as stream:
            stream.write('COMMAND '+repr(args)+'\n'); stream.flush()
            completed = subprocess.run(args, stdout=stream, stderr=subprocess.STDOUT)
        if completed.returncode and check:
            raise RuntimeError(label+' failed; see '+str(LOG))
        return completed.returncode
    def ensure_imports(probe, packages, label):
        import re
        mapping = {p.replace('-', '_'):p for p in packages}
        repaired = set()
        while True:
            offset = LOG.stat().st_size if LOG.exists() else 0
            if not command([sys.executable, '-c', probe], 'Probe '+label, check=False):
                return
            with LOG.open('rb') as stream:
                stream.seek(offset); failure = stream.read().decode('utf-8', errors='replace')
            missing = re.search("ModuleNotFoundError: No module named [\'\"]([^\'\"]+)[\'\"]", failure)
            if "cannot import name 'FqnToConfig'" in failure and 'torchao.quantization' in failure:
                action = 'remove incompatible optional torchao'
                args = [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao']
            elif missing and missing.group(1).split('.')[0] in mapping:
                package = mapping[missing.group(1).split('.')[0]]
                action = 'install missing '+package
                args = [sys.executable, '-m', 'pip', 'install', package]
            else:
                raise RuntimeError(label+' actual import failure; see '+str(LOG))
            if action in repaired:
                raise RuntimeError(label+' still fails after '+action+'; see '+str(LOG))
            repaired.add(action)
            command(args, action)
    ensure_imports('import numpy,torch', ['numpy','torch'], 'tensor dependencies')
    ensure_imports('import accelerate,safetensors,huggingface_hub,transformers,sentencepiece,ftfy; from diffusers import WanPipeline,AutoencoderKLWan',
        ['diffusers','accelerate','safetensors','huggingface-hub','transformers','sentencepiece','ftfy'], 'Wan pipeline and VAE')
    import shutil
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        command(['apt-get','-qq','update'], 'Prepare media dependency')
        command(['apt-get','-qq','install','-y','ffmpeg'], 'Install FFmpeg')
    if not shutil.which('ffmpeg') or not shutil.which('ffprobe'):
        raise RuntimeError('ffmpeg/ffprobe unavailable; see '+str(LOG))

    probe="from pathlib import Path; import importlib; root=Path.cwd().resolve(); names=['main.tube_state.b_rank3_four_source_confirmation_v1','runtime.wan.b_rank3_four_source_confirmation_v1','runtime.wan.b_rank3_four_source_confirmation_backend_v1','runtime.wan.b_rank3_four_source_confirmation_evaluation_v1','experiments.wan_state_clock.b_rank3_four_source_confirmation_v1_run']; modules=[importlib.import_module(n) for n in names]; assert all(Path(m.__file__).resolve().is_relative_to(root) for m in modules); print(*[m.__file__ for m in modules],sep='\\n')"
    subprocess.run([sys.executable,'-c',probe],cwd=SOURCE_ROOT,check=True)
    args=[sys.executable,'-u','-m','experiments.wan_state_clock.b_rank3_four_source_confirmation_v1_run',
          '--input',str(INPUT),'--output',str(OUTPUT)]
    with (OUTPUT/'runner.log').open('a') as stream:
        worker=subprocess.Popen(args,cwd=SOURCE_ROOT,stdout=stream,stderr=subprocess.STDOUT)
        try:
            while worker.poll() is None:
                try:worker.wait(timeout=30)
                except subprocess.TimeoutExpired:
                    progress=OUTPUT/'progress.json'
                    try:
                        value=json.loads(progress.read_text());print(value.get('active'),value.get('counts'),flush=True)
                    except (OSError,ValueError):print('同一worker运行中；详见runner.log。',flush=True)
        except BaseException:
            worker.terminate()
            try:worker.wait(timeout=10)
            except subprocess.TimeoutExpired:worker.kill();worker.wait()
            raise
    print('Runner exit:',worker.returncode,'保存目录:',OUTPUT)


## 方法判断与证据上限

RANK3 至少288实际支持正确槽、每源至少48、至少9阳性DETECT且每源至少2；不得丢失CDC_M实际支持正确槽，错误/误归因/无实际支持输出/null DETECT均为零。全部方法判据满足后，净增至少32槽才宣称固定范围新增贡献。76份直接质量比较只报告数值，不设未采纳质量阈值。null DETECT立即记录候选失败，仅完成既定保存数据诊断。


In [ ]:

recovery=recover_ledger(OUTPUT)
print('恢复后的执行状态:',recovery['counts'],'读取异常数量:',len(recovery['read_errors']))
for name in ('result.json','evaluation.json'):
    try:value=json.loads((OUTPUT/name).read_text())
    except (OSError,ValueError) as error:
        print(name,'尚不可读:',type(error).__name__);continue
    if name=='result.json':
        print('状态:',value.get('status'),'固定分母:',value.get('denominators'))
        print('物理调用:',value.get('physical_counts'),'停止原因:',value.get('stop'))
    else:
        print('方法判据:',value.get('method_status'),value.get('checks'))
        print('新增贡献:',value.get('contribution_status'),'实际支持正确净增:',value.get('net_actual_supported_gain'))
        print('逐源RANK3:',value.get('rank3_per_source'))
        for reader in ('CDC_M','UNION_M','RANK3','RANK2','FIXED_ORIGIN'):
            print(reader,'正例:',value.get('totals',{}).get(reader+'/positive'),'null:',value.get('totals',{}).get(reader+'/null'))
print('完整计划、引用、失败明细和质量记录:',OUTPUT)
print('仅固定四新源和151帧crop/声明的单次删帧；不宣称任意短片、精确seam、来源认证、总体FPR或感知质量PASS。')
